# Song Recs: walkthrough
How a content-based + natural-language recommender works, step by step. Run
`python scripts/build_index.py` once first (downloads the data and embeds every song).

In [1]:
import sys, warnings
sys.path.insert(0, "..")          # so we can import the project package from the notebook
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## Step 1 · Download the dataset from Hugging Face
114k Spotify tracks with audio features. We merge duplicate versions and keep the 30k most popular.

In [2]:
from song_recs.data import load_catalog

df = load_catalog()
print(len(df), "songs")
df[["track_name", "artists", "genres", "popularity", "danceability", "energy", "valence", "tempo"]].head()

30000 songs


,track_name,artists,genres,popularity,danceability,energy,valence,tempo
0,Unholy (feat. Kim Petras),"Sam Smith, Kim Petras","[dance, pop]",100,0.714,0.472,0.238,131.121
1,"Quevedo: Bzrp Music Sessions, Vol. 52","Bizarrap, Quevedo",[hip-hop],99,0.621,0.782,0.550,128.033
2,La Bachata,Manuel Turizo,"[latin, latino, reggae, reggaeton]",98,0.835,0.679,0.850,124.980
3,I'm Good (Blue),"David Guetta, Bebe Rexha","[dance, edm, pop]",98,0.561,0.965,0.304,128.040
4,Me Porto Bonito,"Bad Bunny, Chencho Corleone","[latin, latino, reggae, reggaeton]",97,0.911,0.712,0.425,92.005


## Step 2 · Describe each song in words
Numbers become words people actually use for music, so we can search with plain English.

In [3]:
from song_recs.features import describe

for _, r in df.head(5).iterrows():
    print(f"{r['label'][:45]:47s} → {describe(r)}")

Unholy (feat. Kim Petras) — Sam Smith, Kim Pe   → dance / pop song. Fast, sad, dark, melancholic, danceable, groovy.
Quevedo: Bzrp Music Sessions, Vol. 52 — Bizar   → hip hop song. Fast, high-energy, intense.
La Bachata — Manuel Turizo                      → latin / latino / reggae / reggaeton song. Mid-tempo, happy, upbeat, feel-good, danceable, groovy.
I'm Good (Blue) — David Guetta, Bebe Rexha      → dance / edm / pop song. Fast, high-energy, intense, loud.
Me Porto Bonito — Bad Bunny, Chencho Corleone   → latin / latino / reggae / reggaeton song. Mid-tempo, danceable, groovy.


## Step 3 · Load a sentence-embedding model from Hugging Face
The same model embeds song descriptions *and* your query, so they can be compared directly.

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
v = model.encode(["sad slow acoustic song", "rock song. Slow, sad, melancholic, acoustic.",
                  "edm song. Fast, high-energy, danceable."], normalize_embeddings=True)
print("query vs acoustic song:", round(float(v[0] @ v[1]), 2))
print("query vs edm song:     ", round(float(v[0] @ v[2]), 2))

query vs acoustic song: 0.77
query vs edm song:      0.41


## Step 4 · Recommend from songs you like

In [5]:
from song_recs import Recommender, Request

rec = Recommender()
seeds = [int(rec.search("Hotel California").index[0]), int(rec.search("Bohemian Rhapsody").index[0])]
rec.recommend(Request(liked=seeds, k=8))[["track_name", "artists", "genre", "match", "why"]]

,track_name,artists,genre,match,why
1121,(Everything I Do) I Do It For You,Bryan Adams,rock,0.840962,"similar danceability, acousticness; genre: rock, singer-songwriter"
1300,"Take Me Home, Country Roads",John Denver,country,0.780772,"similar energy; genre: country, folk"
906,Free Fallin',Tom Petty,folk,0.793518,"similar energy, acousticness; genre: folk, rock"
927,Heaven,Bryan Adams,rock,0.782743,"genre: rock, singer-songwriter"
1870,Wrecked,Imagine Dragons,rock,0.745688,similar acousticness; genre: rock
1800,You're the Inspiration - 2006 Remaster,Chicago,folk,0.735832,"similar mood (valence); genre: folk, rock"
327,Fortunate Son,Creedence Clearwater Revival,country,0.698015,"similar acousticness; genre: country, rock"
1002,Brother,Kodaline,rock,0.725461,"similar danceability, tempo; genre: rock"


## Step 5 · …or from a vibe you type

In [6]:
rec.recommend(Request(vibe="calm piano music for focusing", k=8))[["track_name", "artists", "genre", "match"]]

,track_name,artists,genre,match
697,Your Song,Elton John,piano,0.692009
3791,The Longest Time,Billy Joel,piano,0.672922
3806,Hommage d'amitié,Gunnel Boek,piano,0.666770
1007,A Thousand Miles,Vanessa Carlton,piano,0.645743
3064,Quand vous souriez,Libor Kolman,piano,0.665028
8311,The Call,Regina Spektor,piano,0.665815
9716,Candle In The Wind 1997,Elton John,piano,0.662009
4335,Returning,Milo Edale,piano,0.660528


## Step 6 · Diversity: Maximal Marginal Relevance
Compare zero diversity with high diversity: the second list is more varied.

In [7]:
for d in (0.0, 0.7):
    out = rec.recommend(Request(liked=seeds, k=6, diversity=d, max_per_artist=6))
    print(f"diversity={d}:", " | ".join(out["track_name"].str[:22]))

diversity=0.0: (Everything I Do) I Do | Free Fallin' | Heaven | Take Me Home, Country  | Wrecked | You're the Inspiration
diversity=0.7: (Everything I Do) I Do | Demons | Fortunate Son | Take Me Home, Country  | Bad Liar | Believer - Kaskade Rem


## Next
- `python scripts/evaluate.py`: compare against random and most-popular baselines
- `streamlit run app.py`: the interactive version with 👍/👎 feedback and a catalog map